# Monte Carlo Test Results Analysis

This notebook analyzes the results of Monte Carlo tests for inheritance calculations.
Instead of rerunning calculations, it analyzes existing failure reports from tests/output/.

## Available Functions:
- load_latest_failures(): Load the most recent failure data
- get_failure_summary(failures): Get summary statistics
- filter_by_category(failures, category): Filter by ordinary/no_fare/hawashi
- filter_by_heir(failures, heir_name): Filter cases where heir is present
- filter_by_total_range(failures, min_val, max_val): Filter by distribution total
- inspect_case(failure_case): Detailed inspection of a specific case
- find_common_patterns(failures): Identify frequent failure patterns

In [3]:
13 % 12 != 0

True

In [23]:
import math

math.gcd(21,14)

7

In [1]:
from farady import calculate_from_dict

result = calculate_from_dict({'zawja': True, 'umm': 2, 'ab': 3, 'jadd': 2, 'biibn': 2, 'iiibn': 5})

print(result)

[2026-02-26 01:01:47] INFO: === Calculation Started ===
    case_input:
    case: {'iiibn': 5, 'biibn': 2, 'umm': 1, 'ab': 1, 'jadd': 1, 'zawja': 1}
[2026-02-26 01:01:47] INFO: === Calculation Ended ===
    result:
    distribution: {'iiibn': 10, 'biibn': 2, 'umm': 4, 'ab': 4, 'zawja': 3}
    ending: taseeb
    asib: iiibn-biibn
    total: 0.9583333333333334
    status: Unknown
    raas: 24


Case(ibn={}, bint={}, iibn={}, bibn={}, iiibn={'count': 5, 'shares': 10}, biibn={'count': 2, 'shares': 2}, umm={'count': 1, 'fard': Fraction(1, 6), 'shares': 4}, jadda={}, ab={'count': 1, 'fard': Fraction(1, 6), 'shares': 4}, jadd={'count': 1}, lium={}, shaqiqa={}, shaqiq={}, uliab={}, aliab={}, ibnamm_sh={}, ibnamm_liab={}, amm={}, zawj={}, zawja={'count': 1, 'fard': Fraction(1, 8), 'shares': 3}, ending='taseeb', asib='iiibn-biibn', status='Unknown', heads=12, _raas_override=24)


In [2]:
import json
import os
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

# Add the src directory to the path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from farady import calculate_from_dict

# Categories for test cases
CATEGORIES = ("ordinary", "no_fare", "hawashi")

def load_latest_failures(test_name="test_total_always_one"):
    """Load the most recent failure data for a specific test."""
    output_dir = Path("tests/output")
    if not output_dir.exists():
        raise FileNotFoundError(f"Output directory not found at {output_dir}")
        
    # Find all failure files for this test
    failure_files = list(output_dir.glob(f"failures_{test_name}_*.json"))
    if not failure_files:
        raise FileNotFoundError(f"No failure files found for test {test_name}")
        
    # Get the most recent file
    latest_file = sorted(failure_files)[-1]
    print(f"Loading failures from: {latest_file.name}")
    
    with open(latest_file) as f:
        return json.load(f)

def get_failure_summary(failures):
    """Get summary statistics for failure data."""
    print("=== FAILURE SUMMARY ===")
    
    total_failures = 0
    for cat in CATEGORIES:
        cat_failures = failures.get(cat, [])
        count = len(cat_failures)
        total_failures += count
        print(f"{cat.upper()}: {count} failures")
        
        if count > 0:
            # Analyze totals
            totals = [f["result"]["total"] for f in cat_failures]
            print(f"  Total range: {min(totals):.4f} - {max(totals):.4f}")
            print(f"  Average total: {np.mean(totals):.4f}")
            
            # Count by status
            statuses = [f["result"]["status"] for f in cat_failures]
            status_counts = Counter(statuses)
            print(f"  Statuses: {dict(status_counts)}")
            
    print(f"\nTOTAL FAILURES: {total_failures}")

def filter_by_category(failures, category):
    """Filter failures by category (ordinary/no_fare/hawashi)."""
    if category not in CATEGORIES:
        raise ValueError(f"Invalid category. Choose from: {CATEGORIES}")
    return failures.get(category, [])

def filter_by_heir(failures, heir_name):
    """Filter failures where a specific heir is present."""
    filtered = {}
    for cat in CATEGORIES:
        cat_failures = failures.get(cat, [])
        filtered[cat] = [
            f for f in cat_failures 
            if f["case"].get(heir_name, 0) > 0
        ]
    return filtered

def filter_by_total_range(failures, min_val=0.99, max_val=1.01):
    """Filter failures by distribution total range."""
    filtered = {}
    for cat in CATEGORIES:
        cat_failures = failures.get(cat, [])
        filtered[cat] = [
            f for f in cat_failures 
            if min_val <= f["result"]["total"] <= max_val
        ]
    return filtered

def inspect_case(failure_case):
    """Detailed inspection of a specific failure case."""
    print("=== CASE INSPECTION ===")
    print(f"Input case: {failure_case['case']}")
    print(f"Total: {failure_case['result']['total']:.4f}")
    print(f"Status: {failure_case['result']['status']}")
    print(f"Ending: {failure_case['result']['ending']}")
    print(f"Raas: {failure_case['result']['raas']}")
    print(f"Asib: {failure_case['result']['asib']}")
    
    print("\nDistribution:")
    dist = failure_case['result']['distribution']
    for heir, shares in dist.items():
        print(f"  {heir}: {shares} shares")
        
    print("\nNumerators:")
    nums = failure_case['result']['numerators']
    for heir, numerator in nums.items():
        print(f"  {heir}: {numerator}")

def find_common_patterns(failures, top_n=10):
    """Identify frequently occurring case patterns in failures."""
    print("=== COMMON PATTERNS ===")
    
    # Collect all cases
    all_cases = []
    for cat in CATEGORIES:
        cat_failures = failures.get(cat, [])
        all_cases.extend([f["case"] for f in cat_failures])
    
    # Count heir combinations
    pattern_counts = defaultdict(int)
    for case in all_cases:
        # Create a canonical representation of the case
        heirs_present = tuple(sorted([
            (heir, count) for heir, count in case.items() 
            if count > 0 and heir != 'zawj' and heir != 'zawja'
        ]))
        pattern_counts[heirs_present] += 1
        
        # Also count spouses separately since they're boolean
        if case.get('zawj'):
            heirs_present_with_spouse = heirs_present + (('zawj', True),)
            pattern_counts[heirs_present_with_spouse] += 1
        elif case.get('zawja'):
            heirs_present_with_spouse = heirs_present + (('zawja', True),)
            pattern_counts[heirs_present_with_spouse] += 1
    
    # Show top patterns
    sorted_patterns = sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True)
    print(f"Top {min(top_n, len(sorted_patterns))} most common patterns:")
    for i, (pattern, count) in enumerate(sorted_patterns[:top_n]):
        print(f"{i+1:2d}. {dict(pattern)} (appears {count} times)")

In [2]:
# Load failure data
try:
    failures = load_latest_failures("test_total_always_one")
    print("Successfully loaded failure data!\n")
    
    # Get overall summary
    get_failure_summary(failures)
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Make sure you've run the Monte Carlo tests first:")
    print("  poetry run pytest tests/test_monte.py::test_total_always_one")

Loading failures from: failures_test_total_always_one_20260225_232846.json
Successfully loaded failure data!

=== FAILURE SUMMARY ===
ORDINARY: 44 failures
  Total range: 0.0000 - 0.9583
  Average total: 0.4593
  Statuses: {'Failed': 17, 'Unknown': 27}
NO_FARE: 29 failures
  Total range: 0.0000 - 0.9167
  Average total: 0.3994
  Statuses: {'Failed': 10, 'Unknown': 19}
HAWASHI: 70 failures
  Total range: 0.0000 - 0.9167
  Average total: 0.5440
  Statuses: {'Failed': 15, 'Unknown': 55}

TOTAL FAILURES: 143


In [3]:
# Analyze each category separately
print("\n" + "="*50)
print("CATEGORY-BY-CATEGORY ANALYSIS")
print("="*50)

for cat in CATEGORIES:
    print(f"\n--- {cat.upper()} ---")
    cat_failures = failures.get(cat, [])
    
    if not cat_failures:
        print("No failures in this category")
        continue
        
    # Show first few failures
    print(f"First 3 failures (out of {len(cat_failures)}):\n")
    for i, failure in enumerate(cat_failures[:3]):
        print(f"{i+1}. Case: {failure['case']}")
        print(f"   Total: {failure['result']['total']:.4f}")
        print(f"   Status: {failure['result']['status']}")
        print()


CATEGORY-BY-CATEGORY ANALYSIS

--- ORDINARY ---
First 3 failures (out of 44):

1. Case: {}
   Total: 0.0000
   Status: Failed

2. Case: {'zawja': True, 'umm': 2, 'ab': 3, 'jadd': 2, 'biibn': 2, 'iiibn': 5}
   Total: 0.6250
   Status: Unknown

3. Case: {'ibnamm_sh': 0}
   Total: 0.0000
   Status: Failed


--- NO_FARE ---
First 3 failures (out of 29):

1. Case: {}
   Total: 0.0000
   Status: Failed

2. Case: {'zawja': True, 'ab': 0, 'ibnamm_sh': 1, 'aliab': 4, 'uliab': 2, 'shaqiqa': 3}
   Total: 0.9167
   Status: Unknown

3. Case: {'zawj': True, 'jadda': 5, 'shaqiq': 4, 'umm': 2, 'ibnamm_liab': 1, 'shaqiqa': 1}
   Total: 0.6667
   Status: Unknown


--- HAWASHI ---
First 3 failures (out of 70):

1. Case: {}
   Total: 0.0000
   Status: Failed

2. Case: {'ibnamm_liab': 0}
   Total: 0.0000
   Status: Failed

3. Case: {'zawja': True, 'uliab': 1, 'amm': 2, 'shaqiq': 3, 'shaqiqa': 2, 'lium': 0}
   Total: 0.2500
   Status: Unknown



In [4]:
# Detailed inspection of specific cases
print("\n" + "="*50)
print("DETAILED CASE INSPECTION")
print("="*50)

# Inspect the first failure from each category
for cat in CATEGORIES:
    cat_failures = failures.get(cat, [])
    if cat_failures:
        print(f"\n--- First {cat.upper()} failure ---")
        inspect_case(cat_failures[1])
        print("-"*40)


DETAILED CASE INSPECTION

--- First ORDINARY failure ---
=== CASE INSPECTION ===
Input case: {'zawja': True, 'umm': 2, 'ab': 3, 'jadd': 2, 'biibn': 2, 'iiibn': 5}
Total: 0.6250
Status: Unknown
Ending: taseeb
Raas: 24
Asib: iiibn-biibn

Distribution:
  umm: 4 shares
  ab: 4 shares
  jadd: 4 shares
  zawja: 3 shares

Numerators:
  umm: 4
  ab: 4
  jadd: 4
  zawja: 3
----------------------------------------

--- First NO_FARE failure ---
=== CASE INSPECTION ===
Input case: {'zawja': True, 'ab': 0, 'ibnamm_sh': 1, 'aliab': 4, 'uliab': 2, 'shaqiqa': 3}
Total: 0.9167
Status: Unknown
Ending: taseeb
Raas: 60
Asib: aliab-uliab

Distribution:
  shaqiqa: 40 shares
  zawja: 15 shares

Numerators:
  shaqiqa: 40
  zawja: 15
----------------------------------------

--- First HAWASHI failure ---
=== CASE INSPECTION ===
Input case: {'ibnamm_liab': 0}
Total: 0.0000
Status: Failed
Ending: None
Raas: 1
Asib: None

Distribution:

Numerators:
----------------------------------------


In [ ]:
# Find common patterns
print("\n" + "="*50)
print("PATTERN ANALYSIS")
print("="*50)

find_common_patterns(failures)

In [ ]:
# Custom queries
print("\n" + "="*50)
print("CUSTOM QUERIES")
print("="*50)

# Example: Find failures with very low totals
low_total_failures = filter_by_total_range(failures, 0, 0.5)
print("\nFailures with totals < 0.5:")
for cat in CATEGORIES:
    cat_low = low_total_failures.get(cat, [])
    if cat_low:
        print(f"  {cat.upper()}: {len(cat_low)} failures")
        if cat_low:
            # Show one example
            example = cat_low[0]
            print(f"    Example: {example['case']} -> total = {example['result']['total']:.4f}")

# Example: Find failures involving 'ibn'
ibn_failures = filter_by_heir(failures, 'ibn')
print("\nFailures with 'ibn' present:")
for cat in CATEGORIES:
    cat_ibn = ibn_failures.get(cat, [])
    if cat_ibn:
        print(f"  {cat.upper()}: {len(cat_ibn)} failures")
        if cat_ibn:
            # Show one example
            example = cat_ibn[0]
            print(f"    Example: {example['case']} -> ibn gets {example['result']['distribution'].get('ibn', 0)} shares")